# Extract and check shot data for Titanic (1997)
Fetch the Titanic page from Cinemetrics, read its shot statistics, and check ASL. The default uses saved HTML. Set `live_demo = True` for an optional live request.


In [1]:
from zipfile import ZipFile
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import requests

live_demo = False
base = "01_raw/single_film_demo/titanic_1997/"
with ZipFile("data_compact.zip") as data:
    url = json.loads(data.read("02_audit/single_film_demo.json"))["html_source_url"]
    if not live_demo:
        # Follow the capture marker to the exact saved page.
        run = data.read(base + "latest_run.txt").decode().strip()
        html = data.read(base + run + "/cinemetrics.html")
        print("Offline: saved Titanic page")

if live_demo:
    # Keep a new response separate from the frozen capture.
    folder = Path("01_raw/single_film_demo/titanic_1997") / datetime.now(timezone.utc).strftime("live_%Y%m%dT%H%M%S%fZ")
    folder.mkdir(parents=True, exist_ok=False)
    log = {"time_utc": datetime.now(timezone.utc).isoformat(), "url": url, "retry_count": 0}
    clock = time.perf_counter()
    try:
        reply = requests.get(url, timeout=(5, 15), headers={"User-Agent": "ADS206-Titanic-demo/1.0 (educational)"})
        (folder / "cinemetrics.html").write_bytes(reply.content)
        log.update(status=reply.status_code, file="cinemetrics.html", final_url=reply.url)
        print("HTTP status:", reply.status_code)
        reply.raise_for_status()
        html = reply.content
    except requests.RequestException as error:
        log["error"] = str(error)
        raise RuntimeError(f"Live request failed: {error}. Set live_demo = False to use the saved page.") from error
    finally:
        log["seconds"] = round(time.perf_counter() - clock, 3)
        (folder / "requests.jsonl").write_text(json.dumps(log) + "\n", encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")
assert soup.header is not None, "Film header missing"
print("Film:", soup.header.get_text(" ", strip=True))


Offline: saved Titanic page
Film: Titanic ( 1997, USA )


## 2. Extract the shot statistics
Read the four overall statistics from the page buttons. ASL and MSL are in seconds.

In [2]:
# Read overall buttons, not statistics from a scene table.
stats = {}
for button in soup.find_all("button"):
    parts = button.find_all("span", recursive=False)
    if len(parts) == 2:
        label = parts[0].get_text(strip=True).rstrip(":")
        if label in ["NoS", "LEN", "ASL", "MSL"]:
            assert label not in stats, "Repeated overall field"
            stats[label] = parts[1].get_text(strip=True)
assert set(stats) == {"NoS", "LEN", "ASL", "MSL"}, "Missing shot statistics"
print(stats)


{'NoS': '2,673', 'LEN': '03:06:45.6', 'ASL': '4.2', 'MSL': '3.0'}


## 3. Check the ASL arithmetic
Remove the shot-count comma, convert LEN to seconds, then divide by NoS. Compare at the page's one-decimal precision; this does not verify the film edition or playback speed.

In [3]:
# A thousands comma is formatting, not a change in shot count.
shot_count = int(stats["NoS"].replace(",", ""))
parts = stats["LEN"].split(":")
duration_seconds = sum(float(v) * 60 ** i for i, v in enumerate(reversed(parts)))
html_asl = float(stats["ASL"])
assert shot_count > 0 and duration_seconds > 0
calculated_asl = duration_seconds / shot_count
matches = round(calculated_asl, 1) == html_asl
print(f"{duration_seconds:.1f} seconds / {shot_count} shots = {calculated_asl:.2f} seconds per shot")
print("HTML ASL:", html_asl, "match:", matches)
assert matches, "ASL arithmetic differs from the page"


11205.6 seconds / 2673 shots = 4.19 seconds per shot
HTML ASL: 4.2 match: True
